## Print information about a tuned chorale
1. Chorale name, tolerance, tonal diamond shape, limit max
1. Cent values, note names, scores, and ratios for every chord
2. Top notes cents, note names, cent values


In [2]:
import os
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [3]:
import logging, os, sys, time
from importlib import reload
import numpy as np
from importlib import reload
from collections import Counter, defaultdict
user = 'prent'
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')
base_dir = local_dir
WAVE_DIR = os.path.join(base_dir, 'Music', 'sflib')
# The latest files are here: Archive/straw-man/t1_r1.75_s2.50_md28_sn10/bwv253-opt.npy
numpy_dir = os.path.join(base_dir, 'Archive', 'straw-man')

np.set_printoptions(legacy='1.25')
import diamond_music_utils as dmu
import adaptive_tuning_util as atu 
from itertools import count, combinations, permutations
dmu.start_logger('test.log',log_level = 'info') # how to modify this so that it only prints to the log and not in the notebook.
logging.info(f'{base_dir = }, {numpy_dir = }, {WAVE_DIR = }')
rng = np.random.default_rng()

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [4]:
def print_chords(version, input_file, numpy_dir, measure, tolerance, ratios=True, print_individual_chords=True,\
            offset=0, use_werck_top_notes=False, print_top_notes = True):
    
    _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)
    try:
        floating_cents = np.load(input_file)
        existing_chorale_in_cents = np.rint(floating_cents).astype(int)
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    if use_werck_top_notes:
        input_file = os.path.join(numpy_dir, f'{version}-w-top_notes.npy')
    else: 
        input_file = os.path.join(numpy_dir, f'{version}top-notes.npy')
        if print_top_notes:
            print(f'Loaded top_notes from {input_file = }')
    try:      
        top_notes = np.load(input_file)
    except:
        print(f'Trouble loading {input_file = }')
        return {input_file}
    top_notes[1] = top_notes[1] + offset
    
    if print_top_notes:
        print(f'Key: {keys[root]} {mode}, {tolerance = }')
        print(f'\ntop notes:')
        print(*[inx for inx in np.arange(12)], sep='\t')
        print(*[note for note in top_notes[0]], sep = '\t')
        print(*[keys[note] for note in top_notes[0]], sep = '\t')
        print(*[cent_value for cent_value in top_notes[1]], sep = '\t')
    if print_individual_chords: 
        print(f'\n#          cents       note names   chord score')
        # #     +----- cents -----+--- note names---+--- chord score'
        # 0:    0  386    0  884	C♮ E♮ C♮ A♮	47.0
    if measure > 0: print(f'\nprinting only measure {measure}')
    prev_chord = np.zeros(4, dtype=int)
    header1 = f" # Fr/To Cents Ratio\t # Fr/To Cents Ratio\t # Fr/To Cents Ratio"
    
    for inx, chord_in_cents, chord_12 in zip(count(0,1), existing_chorale_in_cents.T, chorale.T):
        if not np.array_equal(prev_chord, chord_in_cents):
            if measure == 0 or 16 * (measure - 1) <= inx < 16 * measure:
                if print_individual_chords: 
                        # Join the note names into a single space-separated string to avoid numpy array formatting
                        pitches = ' '.join(map(str, keys[chord_12 % 12]))
                        print(f'{inx}: {atu.format_chord(chord_in_cents,4)}\t{pitches}\t{chord_scorer.score_chord(chord_in_cents, tolerance=tolerance)}')
                if ratios:
                    print(f'{header1}')
                    intervals = []
                    for inx1, inx2 in combinations(np.arange(4),2):
                            cent_value_interval_pair = np.array([chord_in_cents[inx1], chord_in_cents[inx2]])
                            cent_value_delta, cent_value_moves, cent_value_target = atu.cent_value_interval(cent_value_interval_pair)
                            best_idx = chord_scorer.find_best_interval(cent_value_delta, tolerance)[0]
                            ratio = str(atu.limit_format(tonal_diamond[best_idx])[0]).strip()
                            n1 = keys[chord_12[inx1] % 12]
                            n2 = keys[chord_12[inx2] % 12]
                            intervals.append((n1, n2, cent_value_delta, ratio))

                    def fmt(iv, idx):
                            n1, n2, cents, ratio = iv
                            return f"{idx:>2} {n1:>2} {n2:>2} {cents:>5} {ratio:^6}"

                    # print first and last three intervals on separate lines, nicely aligned and without Python punctuation
                    
                    print("   ".join(fmt(iv, i+1) for i, iv in enumerate(intervals[:3])))
                    print("   ".join(fmt(iv, i+1+3) for i, iv in enumerate(intervals[3:])))
        prev_chord = chord_in_cents.copy()
    return keys, root, mode

In [5]:
print(f'{numpy_dir = }')

numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man'


In [6]:
# python Straw_man_tuning_v3.py --chorale_list bwv253 bwv254 bwv255 bwv256 bwv257 bwv258 bwv259 bwv260 bwv261 bwv262 bwv263 bwv264 --limit_max 17 --tolerance 1 --ratio_factor 1.25 --max_delta 33 --rolls 5 --sa_iters 40 --sa_max_alpha 8.0 --sa_restarts 12 --parallel_restarts 6 --restart_repeat_threshold 2 --no-print_values --numpy_dir Archive/straw-man
# Archive/straw-man/best-tunings/bwv255-trans-sa-opt.npy
ratio_factors = np.array([ "1.25"]) # , "1.25", "1.75"
stability_factors = np.array(["0"]) # , "1.25"
max_delta = 33
snaps = np.array([0])
# suffixes = np.array(['-opt.npy']) # '-opt.npy',
suffixes = np.array(['-trans-sa-opt.npy'])
limit_max = 17
tolerance = 1
measure = 0 # 0 means print all measures
print_individual_chords = False
use_werck_top_notes = False
ratios = False
print_top_notes = False
print_hits_misses = False
total_scores = 0
num_scores = 0
max_score = 0
tonal_diamond = atu.build_tonal_diamond(limit_max)
chord_scorer = atu.ChordScorer(tonal_diamond)

chord_scorer.reset_cache()
for tolerance in [1]:
    for ratio_factor in ratio_factors:
        for stability_factor in stability_factors:
            for snap in snaps:
                for suffix in suffixes:
                    local_numpy_dir = input_file = os.path.join(numpy_dir, f'best-tunings')
                    print(f'{local_numpy_dir = }') 
                    print(f'{tolerance = }, {ratio_factor = }, {stability_factor = }, {snap = }, {suffix = }')
                    for version in ['bwv253', 'bwv254', 'bwv255', 'bwv256', 'bwv257', 'bwv258', 'bwv259', 'bwv260',  'bwv261', 'bwv262', 'bwv263', 'bwv264']: # ['bwv256']: #
                        try:
                            input_file = os.path.join(local_numpy_dir, f'{version}{suffix}') 
                            # print(f'{input_file = }')
                            existing_chorale_in_cents = np.load(input_file)
                            logging.info(f'{input_file = }')
                        except:
                            print(f'Trouble loading {input_file = }')
                            continue
                        num_scores += 1
                        scores = np.array([chord_scorer.score_chord(chord, tolerance=tolerance) for chord in existing_chorale_in_cents.T])
                        print(f'\nversion: {version}, Tol: {tolerance}, RF: {ratio_factor}, Average score: {round(np.average(scores),1)}, max score: {np.max(scores)} max chord: {np.argmax(scores)}')
                        total_scores += np.average(scores)
                        max_score = np.max([max_score, np.max(scores) ])
                        keys, root, mode = print_chords(version, input_file, local_numpy_dir, measure, tolerance, \
                                ratios=ratios, print_individual_chords=print_individual_chords, \
                                use_werck_top_notes=use_werck_top_notes, print_top_notes = print_top_notes)
    if print_hits_misses:
        print(f'hits and misses: {chord_scorer.return_cache_results()}')
    print(f'overall total: {round(total_scores,1)}, {num_scores = }, Average Score: {round(np.average(total_scores/num_scores),1)}, {max_score = }')

local_numpy_dir = '/home/prent/Repos/One-footed-bride-tuning/Archive/straw-man/best-tunings'
tolerance = 1, ratio_factor = '1.25', stability_factor = '0', snap = 0, suffix = '-trans-sa-opt.npy'

version: bwv253, Tol: 1, RF: 1.25, Average score: 48.9, max score: 90.0 max chord: 122

version: bwv254, Tol: 1, RF: 1.25, Average score: 54.8, max score: 145.0 max chord: 158

version: bwv255, Tol: 1, RF: 1.25, Average score: 50.8, max score: 82.0 max chord: 42

version: bwv256, Tol: 1, RF: 1.25, Average score: 53.9, max score: 145.0 max chord: 138

version: bwv257, Tol: 1, RF: 1.25, Average score: 57.2, max score: 140.0 max chord: 128

version: bwv258, Tol: 1, RF: 1.25, Average score: 58.0, max score: 145.0 max chord: 80

version: bwv259, Tol: 1, RF: 1.25, Average score: 53.2, max score: 104.0 max chord: 74

version: bwv260, Tol: 1, RF: 1.25, Average score: 52.2, max score: 105.0 max chord: 72

version: bwv261, Tol: 1, RF: 1.25, Average score: 56.7, max score: 140.0 max chord: 72

version: bw

In [7]:
# Check that cent tunings have not changed the pitch class of any note
print("Checking pitch class preservation...")
violations_found = False
for tolerance in [1]:
    for ratio_factor in ratio_factors:
        for stability_factor in stability_factors:
            for snap in snaps:
                for suffix in suffixes:
                    for version in ['bwv253', 'bwv254', 'bwv255', 'bwv256', 'bwv257', 'bwv258', 'bwv259', 'bwv260', 'bwv261', 'bwv262', 'bwv263', 'bwv264']:
                        try:
                            input_file = os.path.join(local_numpy_dir, f'{version}{suffix}')
                            # input_file = os.path.join(local_numpy_dir, f'{version}-trans-sa-opt.npy') # -trans-sa-opt.npy
                            # print(f'{input_file = }')
                            existing_chorale_in_cents = np.load(input_file)
                        except:
                            print(f'Could not load {input_file}')
                            continue

                        _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)

                        violations = []
                        for chord_inx, (chord_in_cents, chord_12) in enumerate(zip(existing_chorale_in_cents.T, chorale.T)):
                            for voice, (cents, midi) in enumerate(zip(chord_in_cents, chord_12)):
                                original_pc = int(midi) % 12
                                tuned_pc = int(atu.pitch_class_from_cents(cents))  # half-up rounding, consistent with horizontal_transpose.py
                                if original_pc != tuned_pc:
                                    violations.append((chord_inx, voice, int(midi), cents, original_pc, tuned_pc))

                        if violations:
                            violations_found = True
                            print(f'\n{version}: {len(violations)} pitch class violation(s):')
                            for chord_inx, voice, midi, cents, orig_pc, tuned_pc in violations:
                                print(f'  chord {chord_inx}, voice {voice}: MIDI {midi} ({keys[orig_pc]}) -> {cents} cents ({keys[tuned_pc]})')
                        else:
                            print(f'{version}: OK')

if not violations_found:
    print('\nAll pitch classes preserved across all chorales.')

Checking pitch class preservation...
bwv253: OK
bwv254: OK
bwv255: OK
bwv256: OK
bwv257: OK
bwv258: OK
bwv259: OK
bwv260: OK
bwv261: OK
bwv262: OK
bwv263: OK
bwv264: OK

All pitch classes preserved across all chorales.


In [8]:
# Analyze the instrument section feature arrays, created by WreckingCrew.py, which will eventually be used by csound to create the audio output. Each section has its own feature array, which is created by the woodwinds_part, finger_piano_part, melody_part, or the bass_part functions. These four functions are used by different sections of the orchestra. Pick the one you want by setting the feature_array in the next line 


features = {0: 'instrument', 1: 'duration before next note', 2: 'hold_time', 3: 'velocity', 4: 'cents', 5: 'octave', 6: 'voice', 7: 'stereo', 8: 'envelope', 9: 'glissando', 10: 'upsample', 11: 'right_side_envelope', 12: 'second_glissando', 13: 'third_glissando', 14: 'volume'}

all_feature_array_names = ['perc_part_finger_pianos.npy', 'perc_part_pizz_strings.npy', 'perc_part_perc_guitar.npy', 'bass_part_bass_section.npy' , 'melody_part_melody_section.npy', 'winds_part_wood_winds.npy', 'winds_part_brass_section.npy', 'winds_part_bowed_strings.npy']

all_feature_array_names = ['bass_part_bass_section.npy'] # just pick one for now

for feature_array_name in all_feature_array_names:
    feature_array = np.load(feature_array_name, allow_pickle=True)
    # the column locations of the features in the feature_array. 
    
    print(f'section name: {feature_array_name}, note count: {feature_array.shape[0]}')
    for feature in [14, 3]:
        vals, counts = np.unique(np.round(feature_array[:, feature], 2), return_counts=True)
        pairs = [(f'{v:.2f}', int(c)) for v, c in zip(vals, counts)]
        print(f'{features[feature]} values & counts: {pairs}')
        print(f'average: {np.round(np.average(feature_array[:, feature]), 2)}, std: {np.round(np.std(feature_array[:, feature]), 2 )}, min: {np.round(np.min(feature_array[:, feature]), 2)}, max: {np.round(np.max(feature_array[:, feature]), 2)}')
        print(f'') 

section name: bass_part_bass_section.npy, note count: 5160
volume values & counts: [('0.00', 225), ('0.05', 478), ('0.27', 182), ('1.05', 206), ('1.07', 416), ('1.48', 83), ('1.73', 414), ('1.81', 247), ('2.00', 43), ('2.21', 239), ('2.47', 90), ('2.89', 367), ('2.92', 211), ('3.01', 210), ('3.95', 124), ('4.00', 87), ('4.05', 176), ('4.27', 67), ('5.05', 76), ('5.07', 143), ('5.09', 160), ('5.23', 159), ('5.48', 27), ('5.73', 116), ('5.81', 84), ('6.00', 15), ('6.21', 85), ('6.47', 29), ('6.89', 110), ('6.92', 76), ('7.01', 58), ('7.95', 45), ('9.09', 62), ('9.23', 50)]
average: 2.89, std: 2.26, min: 0.0, max: 9.23

velocity values & counts: [('68.00', 565), ('70.00', 727), ('71.00', 1257), ('73.00', 1426), ('74.00', 495), ('76.00', 690)]
average: 72.04, std: 2.31, min: 68.0, max: 76.0



In [ ]:
# This cell is specifically targeted at volume and velocity. I want to know how csound will process those features. Looking at this line of csound code: 
#       iamp = ampdb(p4) * p15 / 5 
# that is how csound processes the velocity and volume features. In the features numpy array, those are actually index 14 for volume and 3 for velocity, even though they are 4 and 15 in the csound code. I want to know what each note value for that combination of features using that calculation is actually producing. 

features = {0: 'instrument', 1: 'duration before next note', 2: 'hold_time', 3: 'velocity', 4: 'cents', 5: 'octave', 6: 'voice', 7: 'stereo', 8: 'envelope', 9: 'glissando', 10: 'upsample', 11: 'right_side_envelope', 12: 'second_glissando', 13: 'third_glissando', 14: 'volume'}

all_feature_array_names = ['perc_part_finger_pianos.npy', 'perc_part_pizz_strings.npy', 'perc_part_perc_guitar.npy', 'bass_part_bass_section.npy' , 'melody_part_melody_section.npy', 'winds_part_wood_winds.npy', 'winds_part_brass_section.npy', 'winds_part_bowed_strings.npy']

all_feature_array_names = ['bass_part_bass_section.npy'] # just pick one for now

for feature_array_name in all_feature_array_names:
    feature_array = np.load(feature_array_name, allow_pickle=True)
    # the column locations of the features in the feature_array. 
    
    print(f'section name: {feature_array_name}, note count: {feature_array.shape[0]}')
    for feature in [14, 3]:
        vals, counts = np.unique(np.round(feature_array[:, feature], 2), return_counts=True)
        pairs = [(f'{v:.2f}', int(c)) for v, c in zip(vals, counts)]
        print(f'{features[feature]} values & counts: {pairs}')
        print(f'average: {np.round(np.average(feature_array[:, feature]), 2)}, std: {np.round(np.std(feature_array[:, feature]), 2 )}, min: {np.round(np.min(feature_array[:, feature]), 2)}, max: {np.round(np.max(feature_array[:, feature]), 2)}')
        print(f'') 

section name: bass_part_bass_section.npy, note count: 5160
volume values & counts: [('0.00', 225), ('0.05', 478), ('0.27', 182), ('1.05', 206), ('1.07', 416), ('1.48', 83), ('1.73', 414), ('1.81', 247), ('2.00', 43), ('2.21', 239), ('2.47', 90), ('2.89', 367), ('2.92', 211), ('3.01', 210), ('3.95', 124), ('4.00', 87), ('4.05', 176), ('4.27', 67), ('5.05', 76), ('5.07', 143), ('5.09', 160), ('5.23', 159), ('5.48', 27), ('5.73', 116), ('5.81', 84), ('6.00', 15), ('6.21', 85), ('6.47', 29), ('6.89', 110), ('6.92', 76), ('7.01', 58), ('7.95', 45), ('9.09', 62), ('9.23', 50)]
average: 2.89, std: 2.26, min: 0.0, max: 9.23

velocity values & counts: [('68.00', 565), ('70.00', 727), ('71.00', 1257), ('73.00', 1426), ('74.00', 495), ('76.00', 690)]
average: 72.04, std: 2.31, min: 68.0, max: 76.0



In [44]:
for quantization in [3,7,8,9]:
    tempo_sum = 0
    for repeats_average in [3, 14, 10, 9, 15, 4, 14, 36, 11, 13, 21, 27]:
        if repeats_average == 2:
            tempo = rng.choice(np.arange(30, 40, 4))
        elif repeats_average * quantization > 65:
            tempo = rng.choice(np.arange(106, 124, 4)) 
        elif repeats_average * quantization > 40:
            tempo = rng.choice(np.arange(80, 105, 4)) 
        elif repeats_average * quantization > 30:
            tempo = rng.choice(np.arange(60, 80, 4)) 
        else: tempo = rng.choice(np.arange(56, 63, 4))
        tempo_sum += tempo
        # print(f'{repeats_average = }, {repeats_average * quantization}, {tempo = }')
    print(f'{quantization = }, {round(tempo_sum / 12,2) = }')

quantization = 3, round(tempo_sum / 12,2) = 80.0
quantization = 7, round(tempo_sum / 12,2) = 101.83
quantization = 8, round(tempo_sum / 12,2) = 104.0
quantization = 9, round(tempo_sum / 12,2) = 105.33
